In [5]:
import os
import numpy as np
import nibabel as nib
import pandas as pd
from skimage.measure import label

# ========================= RUTAS =========================
EXCEL_PATH = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/CSIRO_dataset/rCMBInformationInfo.xlsx"
MASK_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/nnUNet_results_test_with_rCMBs" 
OUTPUT_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/check2_nnUNet_results_test_with_rCMBs"

os.makedirs(OUTPUT_DIR, exist_ok=True)

def safe_int_minus1(val):
    if pd.isna(val) or str(val).strip() == '': return None
    try:
        raw = int(float(str(val).strip()))
        return raw - 1 if raw >= 0 else None # Ajustado para permitir index 0
    except: return None

# ========================= PROCESAMIENTO =========================
df = pd.read_excel(EXCEL_PATH)
results_list = []
total_rcmbs_real_global = 0

for idx, row in df.iterrows():
    nifti_name = str(row.iloc[0]).strip()
    if not nifti_name.endswith('.nii.gz'): continue

    mask_path = os.path.join(MASK_DIR, nifti_name)
    if not os.path.exists(mask_path): continue

    mask_data = nib.load(mask_path).get_fdata()
    shape = mask_data.shape

    # 1. Extraer coordenadas reales (Ground Truth)
    coords_gt = []
    for i in range(1, len(row), 3):
        x, y, z = safe_int_minus1(row.iloc[i]), safe_int_minus1(row.iloc[i+1]), safe_int_minus1(row.iloc[i+2])
        if all(v is not None for v in [x, y, z]):
            if 0 <= x < shape[0] and 0 <= y < shape[1] and 0 <= z < shape[2]:
                coords_gt.append((x, y, z))
    
    total_rcmbs_real_global += len(coords_gt)

    # 2. Etiquetar detecciones del modelo (sCMBs detectados)
    labeled_mask, num_sCMB_total = label(mask_data > 0, return_num=True)
    
    # 3. Verificación con vecindario (3x3x3)
    blobs_hit = set()
    tp_count = 0
    
    for (cx, cy, cz) in coords_gt:
        found_in_neighborhood = False
        # Buscar en un cubo de 3x3x3 alrededor del centroide
        for dx in [-1, 0, 1]:
            for dy in [-1, 0, 1]:
                for dz in [-1, 0, 1]:
                    nx, ny, nz = cx + dx, cy + dy, cz + dz
                    if 0 <= nx < shape[0] and 0 <= ny < shape[1] and 0 <= nz < shape[2]:
                        blob_id = labeled_mask[nx, ny, nz]
                        if blob_id > 0:
                            blobs_hit.add(blob_id)
                            found_in_neighborhood = True
        if found_in_neighborhood:
            tp_count += 1

    # Cálculos de sCMB (Detecciones del modelo)
    scmb_correctos = len(blobs_hit) 
    scmb_erroneos = num_sCMB_total - scmb_correctos
    fn = len(coords_gt) - tp_count

    # Métricas
    precision = tp_count / num_sCMB_total if num_sCMB_total > 0 else 0
    recall = tp_count / len(coords_gt) if len(coords_gt) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    results_list.append({
        'ID': nifti_name,
        'rCMBs_Reales': len(coords_gt),
        'sCMBs_Detectados_Total': num_sCMB_total,
        'sCMBs_Correctos(TP)': scmb_correctos,
        'sCMBs_Erroneos(FP)': scmb_erroneos,
        'Missed(FN)': fn,
        'Precision': round(precision, 4),
        'Recall': round(recall, 4),
        'F1-Score': round(f1, 4)
    })

# ========================= SALIDA =========================
df_res = pd.DataFrame(results_list)
csv_path = os.path.join(OUTPUT_DIR, "analisis_completo_CMB.csv")
df_res.to_csv(csv_path, index=False)

total_scmb_gen = df_res['sCMBs_Detectados_Total'].sum()
total_scmb_corr = df_res['sCMBs_Correctos(TP)'].sum()
total_scmb_err = df_res['sCMBs_Erroneos(FP)'].sum()

print("-" * 40)
print(f"ANÁLISIS FINALIZADO")
print(f"Total rCMBs (Reales en Excel): {total_rcmbs_real_global}")
print(f"Total sCMBs (Generados por el modelo): {total_scmb_gen}")
print(f"  > sCMBs Correctos (TP): {total_scmb_corr}")
print(f"  > sCMBs Erróneos (FP): {total_scmb_err}")
print("-" * 40)
print(f"Media Recall (Sensibilidad): {df_res['Recall'].mean():.4f}")
print(f"Media Precision: {df_res['Precision'].mean():.4f}")
print(f"Media F1-Score: {df_res['F1-Score'].mean():.4f}")
print(f"Resultados guardados en: {csv_path}")

----------------------------------------
ANÁLISIS FINALIZADO
Total rCMBs (Reales en Excel): 146
Total sCMBs (Generados por el modelo): 110
  > sCMBs Correctos (TP): 60
  > sCMBs Erróneos (FP): 50
----------------------------------------
Media Recall (Sensibilidad): 0.4179
Media Precision: 0.4631
Media F1-Score: 0.4140
Resultados guardados en: /media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/check2_nnUNet_results_test_with_rCMBs/analisis_completo_CMB.csv


Ampliamos un poco este análisis para entender a qué se debe los fallos.

In [11]:
import os
import numpy as np
import nibabel as nib
import pandas as pd
from skimage.measure import label, regionprops
import matplotlib.pyplot as plt

# ========================= RUTAS =========================
EXCEL_PATH = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/CSIRO_dataset/rCMBInformationInfo.xlsx"
MASK_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/nnUNet_results_test_with_rCMBs" 
SWI_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/CSIRO_Real_CMBs_Test/imagesTs" 
OUTPUT_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/check2_nnUNet_results_test_with_rCMBs"
DIAG_DIR = os.path.join(OUTPUT_DIR, "diagnostico_visual")

os.makedirs(DIAG_DIR, exist_ok=True)

def safe_int_minus1(val):
    if pd.isna(val) or str(val).strip() == '': return None
    try:
        raw = int(float(str(val).strip()))
        return raw - 1 if raw >= 0 else None
    except: return None

# ========================= PROCESAMIENTO =========================
df = pd.read_excel(EXCEL_PATH)
results_list = []
detailed_analysis = [] 
total_rcmbs_real_global = 0

for idx, row in df.iterrows():
    raw_name = str(row.iloc[0]).strip()
    # Nombre para la MÁSCARA (sin _0000)
    nifti_name = raw_name if raw_name.endswith('.nii.gz') else raw_name + '.nii.gz'
    
    # Nombre para la IMAGEN SWI (con _0000) -> added by me
    base_name = nifti_name.replace(".nii.gz", "")
    swi_name = f"{base_name}_0000.nii.gz"
    
    mask_path = os.path.join(MASK_DIR, nifti_name)
    swi_path = os.path.join(SWI_DIR, swi_name)
    
    if not os.path.exists(mask_path):
        continue

    print(f"Procesando: {nifti_name} (Imagen: {swi_name})")

    mask_data = nib.load(mask_path).get_fdata()
    
    # Cargar SWI con el nuevo nombre
    swi_data = None
    if os.path.exists(swi_path):
        swi_data = nib.load(swi_path).get_fdata()
    else:
        print(f"ADVERTENCIA: No se encontró la imagen SWI en {swi_path}")
        
    shape = mask_data.shape

    # 1. Coordenadas Reales
    coords_gt = []
    for i in range(1, len(row), 3):
        try:
            x, y, z = safe_int_minus1(row.iloc[i]), safe_int_minus1(row.iloc[i+1]), safe_int_minus1(row.iloc[i+2])
            if all(v is not None for v in [x, y, z]):
                if 0 <= x < shape[0] and 0 <= y < shape[1] and 0 <= z < shape[2]:
                    coords_gt.append((x, y, z))
        except: break
    
    total_rcmbs_real_global += len(coords_gt)
    labeled_mask, num_sCMB_total = label(mask_data > 0, return_num=True)
    
    blobs_hit = set()
    tp_count = 0
    
    for c_idx, (cx, cy, cz) in enumerate(coords_gt):
        found_in_neighborhood = False
        for dx in [-1, 0, 1]:
            for dy in [-1, 0, 1]:
                for dz in [-1, 0, 1]:
                    nx, ny, nz = cx + dx, cy + dy, cz + dz
                    if 0 <= nx < shape[0] and 0 <= ny < shape[1] and 0 <= nz < shape[2]:
                        if labeled_mask[nx, ny, nz] > 0:
                            blobs_hit.add(labeled_mask[nx, ny, nz])
                            found_in_neighborhood = True
        
        tipo = "TP" if found_in_neighborhood else "FN"
        if found_in_neighborhood: tp_count += 1

        # Análisis cuantitativo de intensidad -> added by me
        if swi_data is not None:
            detailed_analysis.append({
                'ID': nifti_name, 
                'tipo': tipo, 
                'intensidad': swi_data[cx, cy, cz],
                'coord': f"{cx}_{cy}_{cz}"
            })
            
            # Generar parches visuales para el estudio de fallos
            if tipo == "FN":
                # Tomamos un recorte de 40x40 en el plano axial
                z_slice = swi_data[:, :, cz]
                patch = z_slice[max(0, cx-20):min(shape[0], cx+20), max(0, cy-20):min(shape[1], cy+20)]
                plt.imsave(os.path.join(DIAG_DIR, f"FN_{base_name}_c{c_idx}.png"), patch, cmap='gray')

    # Falsos Positivos
    if swi_data is not None:
        props = regionprops(labeled_mask)
        for p in props:
            if p.label not in blobs_hit:
                bx, by, bz = map(int, p.centroid)
                detailed_analysis.append({
                    'ID': nifti_name, 
                    'tipo': 'FP', 
                    'intensidad': swi_data[bx, by, bz], 
                    'size_voxels': p.area
                })

    # Métricas (Tu lógica original)
    scmb_correctos = len(blobs_hit) 
    scmb_erroneos = num_sCMB_total - scmb_correctos
    fn_count = len(coords_gt) - tp_count

    precision = tp_count / num_sCMB_total if num_sCMB_total > 0 else 0
    recall = tp_count / len(coords_gt) if len(coords_gt) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    results_list.append({
        'ID': nifti_name, 'rCMBs_Reales': len(coords_gt), 'sCMBs_Detectados_Total': num_sCMB_total,
        'sCMBs_Correctos(TP)': scmb_correctos, 'sCMBs_Erroneos(FP)': scmb_erroneos, 'Missed(FN)': fn_count,
        'Precision': round(precision, 4), 'Recall': round(recall, 4), 'F1-Score': round(f1, 4)
    })

# ========================= SALIDA =========================
df_res = pd.DataFrame(results_list)
df_res.to_csv(os.path.join(OUTPUT_DIR, "analisis_completo_CMB.csv"), index=False)

if detailed_analysis:
    df_det = pd.DataFrame(detailed_analysis)
    df_det.to_csv(os.path.join(OUTPUT_DIR, "analisis_intensidades_diagnostico.csv"), index=False)

print("-" * 40)
print(f"ANÁLISIS FINALIZADO")
print(f"Archivos procesados: {len(results_list)}")
print(f"Media Recall: {df_res['Recall'].mean():.4f}")
print(f"Datos de intensidad guardados en: analisis_intensidades_diagnostico.csv")

Procesando: 122_T1_MRI_SWI_BFC_50mm_HM.nii.gz (Imagen: 122_T1_MRI_SWI_BFC_50mm_HM_0000.nii.gz)
Procesando: 122_T2_MRI_SWI_BFC_50mm_HM.nii.gz (Imagen: 122_T2_MRI_SWI_BFC_50mm_HM_0000.nii.gz)
Procesando: 218_T0_MRI_SWI_BFC_50mm_HM.nii.gz (Imagen: 218_T0_MRI_SWI_BFC_50mm_HM_0000.nii.gz)
Procesando: 218_T2_MRI_SWI_BFC_50mm_HM.nii.gz (Imagen: 218_T2_MRI_SWI_BFC_50mm_HM_0000.nii.gz)
Procesando: 218_T3_MRI_SWI_BFC_50mm_HM.nii.gz (Imagen: 218_T3_MRI_SWI_BFC_50mm_HM_0000.nii.gz)
Procesando: 219_T0_MRI_SWI_BFC_50mm_HM.nii.gz (Imagen: 219_T0_MRI_SWI_BFC_50mm_HM_0000.nii.gz)
Procesando: 221_T2_MRI_SWI_BFC_50mm_HM.nii.gz (Imagen: 221_T2_MRI_SWI_BFC_50mm_HM_0000.nii.gz)
Procesando: 222_T0_MRI_SWI_BFC_50mm_HM.nii.gz (Imagen: 222_T0_MRI_SWI_BFC_50mm_HM_0000.nii.gz)
Procesando: 222_T1_MRI_SWI_BFC_50mm_HM.nii.gz (Imagen: 222_T1_MRI_SWI_BFC_50mm_HM_0000.nii.gz)
Procesando: 238_T0_MRI_SWI_BFC_50mm_HM.nii.gz (Imagen: 238_T0_MRI_SWI_BFC_50mm_HM_0000.nii.gz)
Procesando: 238_T1_MRI_SWI_BFC_50mm_HM.nii.gz (Ima

In [13]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import numpy as np
import nibabel as nib

# ========================= CONFIGURACIÓN =========================
CSV_PATH = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/check2_nnUNet_results_test_with_rCMBs/analisis_intensidades_diagnostico.csv"
OUTPUT_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/check2_nnUNet_results_test_with_rCMBs/informe_final"
SWI_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/CSIRO_Real_CMBs_Test/imagesTs" 

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "galeria_deteccion"), exist_ok=True)

# 1. GENERACIÓN DEL INFORME CUANTITATIVO
df = pd.read_csv(CSV_PATH)

plt.figure(figsize=(10, 6))
sns.boxplot(x='tipo', y='intensidad', data=df, palette='viridis')
plt.title('Comparativa de Intensidad en el Centroide: TP vs FN vs FP')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.savefig(os.path.join(OUTPUT_DIR, "boxplot_intensidades.png"), dpi=300)
plt.close()

# Estadísticas rápidas
stats = df.groupby('tipo')['intensidad'].describe()
stats.to_csv(os.path.join(OUTPUT_DIR, "resumen_estadistico.csv"))

print("Informe cuantitativo generado.")

# 2. GENERACIÓN DE PARCHES DE ALTA CALIDAD (No borrosos)
def save_enhanced_patch(image_data, coords, name, folder, window=30):
    cx, cy, cz = coords
    # Extraer plano axial
    z_slice = image_data[:, :, int(cz)]
    
    # Recorte con seguridad de bordes
    x_start, x_end = max(0, int(cx)-window), min(image_data.shape[0], int(cx)+window)
    y_start, y_end = max(0, int(cy)-window), min(image_data.shape[1], int(cy)+window)
    patch = z_slice[x_start:x_end, y_start:y_end]
    
    # Crear figura para evitar el emborronamiento del visualizador de fotos
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(patch, cmap='gray', interpolation='nearest') # 'nearest' evita el difuminado
    
    # Dibujar cruceta en el centro del parche (que es window, window si no hay bordes)
    center_x = int(cx) - x_start
    center_y = int(cy) - y_start
    ax.axhline(center_y, color='red', linewidth=0.8, alpha=0.6)
    ax.axvline(center_x, color='red', linewidth=0.8, alpha=0.6)
    
    ax.set_title(f"Tipo: {name}")
    ax.axis('off')
    
    plt.savefig(os.path.join(folder, f"{name}.png"), bbox_inches='tight', dpi=150)
    plt.close()

# Generar una muestra de 10 FNs y 10 FPs para inspección manual
for tipo in ['FN', 'FP']:
    sub_df = df[df['tipo'] == tipo].head(10)
    for i, row in sub_df.iterrows():
        # Re-formatear nombre para encontrar la imagen
        base_name = row['ID'].replace(".nii.gz", "")
        swi_path = os.path.join(SWI_DIR, f"{base_name}_0000.nii.gz")
        
        if os.path.exists(swi_path):
            img = nib.load(swi_path).get_fdata()
            # Recuperar coordenadas (están guardadas como string en el CSV previo)
            if 'coord' in row and pd.notna(row['coord']):
                c = [float(x) for x in str(row['coord']).split('_')]
                save_enhanced_patch(img, c, f"{tipo}_{base_name}_idx{i}", 
                                   os.path.join(OUTPUT_DIR, "galeria_deteccion"))

print(f"Proceso finalizado. Revisa la carpeta: {OUTPUT_DIR}")

/tmp/ipykernel_78948/868262627.py:20: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x='tipo', y='intensidad', data=df, palette='viridis')


Informe cuantitativo generado.
Proceso finalizado. Revisa la carpeta: /media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/check2_nnUNet_results_test_with_rCMBs/informe_final
